# ClauseAnchor: Colab environment check
Owner: Tanishq Arya. Run after committing the Phase 1 files to GitHub.

This notebook clones a specific commit, installs the small core package and runs its sample tests. It does not train or download a model. A CPU runtime is sufficient. Run cells in order with Shift+Enter. Code remains in `ml/`; the notebook is a launcher.


In [ ]:
import sys
from pathlib import Path
import subprocess
print(sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This project requires Python 3.12. Select a compatible runtime before continuing; do not silently change the project requirement.")


## Choose the exact source revision
On your laptop, after pushing, run `git rev-parse HEAD` and paste its 40-character result below. The clone URL has no embedded credentials. This cell is for the public repository. If access is private, stop and configure an approved authenticated access method without putting tokens in cells or URLs.


In [ ]:
REPOSITORY = "https://github.com/TheMEGALODON55681/clauseanchor.git"
COMMIT = input("Paste the pushed 40-character Git commit: ").strip()
if len(COMMIT) != 40 or any(ch not in "0123456789abcdef" for ch in COMMIT):
    raise ValueError("Expected a lowercase 40-character commit hash.")
repo = Path("/content/clauseanchor")
if not repo.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(repo)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError("Existing folder has a different origin. Inspect it before continuing.")
    dirty = subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Existing clone has edits. Save and review them before switching revisions.")
    subprocess.run(["git", "-C", str(repo), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", COMMIT], check=True)
if not (repo / "ml/pyproject.toml").is_file():
    raise RuntimeError("This commit lacks the Phase 1 package. Push it from your laptop first.")
print("Using commit:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo / "ml")], check=True)
subprocess.run([sys.executable, str(repo / "ml/scripts/check_contract.py")], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(repo / "ml/tests"), "-v"], cwd=repo, check=True)


## Inspect compute
A CPU is enough for this notebook. For later training, use Runtime > Change runtime type and select an available GPU, reconnect, then run again. Availability varies. No model is loaded by this cell.


In [ ]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except ImportError:
    print("PyTorch is not installed. Phase 1 does not need it. Install a verified training environment in Phase 2.")


## Optional: prepare persistent project folders
Run only if you want to connect your own Google Drive. Follow the Google account chooser. This creates project folders and writes an environment-check record, without model weights. Persistent checkpoints will be added by the later training runner. A saved notebook alone does not save a running model.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
project_storage = Path("/content/drive/MyDrive/ClauseAnchor")
for name in ("data", "checkpoints", "exports", "logs"):
    (project_storage / name).mkdir(parents=True, exist_ok=True)
import datetime
import json
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
record = {"kind": "environment_check_only", "python": sys.version, "git_commit": COMMIT, "utc": stamp}
path = project_storage / "logs" / f"environment-{stamp}.json"
path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("Saved environment record:", path)


## Done
The expected result is a contract PASS and 26 passing tests. This proves the starter runs in this environment. It does not prove GPU training, dataset correctness or model quality. Keep notebook outputs clear before committing. Continue with data manifests and baseline reproduction in the guide.
